# Module 3c — Build a BioASQ weight-calibration dataset (optional)

**Time:** varies · **Input:** `data/BioASQ/questions_all_164.csv` and your MyGPT library · **Output:** `workshop_outputs/03_calibration_eval_dataset.csv`

This optional notebook uses the original BioASQ questions and ground truths instead of generating new questions. It runs each question through MyGPT, then uses the LLM judge validated in 03b to label correctness. It does not repeat human labeling or calculate Cohen’s κ.

The source file is named `questions_all_164.csv`; the current file contains 163 questions. Ground truths include yes/no answers and factoid answer lists.

In [1]:
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd

# Make the repository root importable whether Jupyter was started at the root or here.
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir  # noqa: E402

OUT = outputs_dir()


In [2]:
QUESTIONS_FILE = ROOT / "data" / "BioASQ" / "questions_all_164.csv"
ANSWERS_FILE = OUT / "03_calibration_answers.csv"
JUDGE_PROGRESS_FILE = OUT / "03_calibration_judge_progress.csv"
EVAL_FILE = OUT / "03_calibration_eval_dataset.csv"
JUDGE_LLM = os.environ.get("JUDGE_LLM")

if not QUESTIONS_FILE.exists():
    raise FileNotFoundError(f"BioASQ source file not found: {QUESTIONS_FILE}")

# This source CSV contains a cp1252 character in a question.
questions = pd.read_csv(QUESTIONS_FILE, encoding="cp1252")
required = {"bioasq_id", "questions", "ground_truth"}
missing = required - set(questions.columns)
if missing:
    raise ValueError(f"BioASQ CSV is missing required columns: {sorted(missing)}")
if questions[["bioasq_id", "questions", "ground_truth"]].isna().any().any():
    raise ValueError("BioASQ source contains a missing ID, question, or ground truth")
if questions["bioasq_id"].duplicated().any():
    raise ValueError("BioASQ source contains duplicate bioasq_id values")

questions = questions.rename(columns={"questions": "question", "ground_truth": "reference_answer"})
questions.insert(0, "question_id", range(1, len(questions) + 1))
questions["answerable"] = 1
print(f"Loaded {len(questions)} BioASQ questions from {QUESTIONS_FILE.name}")
questions[["question_id", "bioasq_id", "question", "reference_answer"]].head()


Loaded 163 BioASQ questions from questions_all_164.csv


,question_id,bioasq_id,question,reference_answer
0,1,67fc20ee18b1e36f2e00012a,Do glioblastoma tumors have a high degree of l...,no
1,2,67ed650c18b1e36f2e0000fe,Has any single infectious agent been identifie...,no
2,3,67daf71018b1e36f2e000057,Is pollinosis another name for allergies to tr...,yes
3,4,67d4903418b1e36f2e000018,Which gene shapes the 3D enhancer network of t...,[['SATB1']]
4,5,67f6f14118b1e36f2e000100,Do mesenchymal-like glioblastoma cells show in...,yes


## 1. Run every BioASQ question through MyGPT

This can take a while. Results are checkpointed after each question. If the cell is rerun, completed answers for the same BioASQ IDs are reused and failed rows are retried. The source ground truth is preserved in `reference_answer`.

In [4]:
client = MyGPTClient.from_env()
previous = {}
if ANSWERS_FILE.exists():
    cached = pd.read_csv(ANSWERS_FILE)
    if "bioasq_id" in cached.columns and "answer" in cached.columns:
        previous = {str(row.bioasq_id): row.to_dict() for _, row in cached.iterrows()}

answers = []
for _, row in questions.iterrows():
    bioasq_id = str(row.bioasq_id)
    cached_row = previous.get(bioasq_id)
    cache_error = cached_row.get("query_error") if cached_row else None
    cache_matches = cached_row and str(cached_row.get("question", "")) == str(row.question) and str(cached_row.get("reference_answer", "")) == str(row.reference_answer)
    if cache_matches and str(cached_row.get("answer", "")).strip() and not (pd.notna(cache_error) and str(cache_error).strip()):
        answers.append(cached_row)
        print(f"[{row.question_id}/{len(questions)}] cached: {row.question}")
        continue

    try:
        result = client.ask(str(row.question))
        record = {**row.to_dict(), **result, "query_error": ""}
        if not str(record.get("answer", "")).strip():
            record["query_error"] = "MyGPT returned an empty answer"
    except Exception as exc:
        record = {**row.to_dict(), "answer": "", "query_error": str(exc)}
        print(f"[{row.question_id}] MyGPT request failed: {exc}")

    answers.append(record)
    pd.DataFrame(answers).to_csv(ANSWERS_FILE, index=False)
    print(f"[{row.question_id}/{len(questions)}] completed")

answers = pd.DataFrame(answers)
answers.to_csv(ANSWERS_FILE, index=False)
failed = answers[answers["query_error"].fillna("").ne("")]
print(f"Saved {len(answers) - len(failed)}/{len(questions)} MyGPT answers to {ANSWERS_FILE}")
if not failed.empty:
    print("Rerun this cell to retry failed questions:", failed["bioasq_id"].tolist())


[1/163] cached: Do glioblastoma tumors have a high degree of lymphocyte infiltration?
[2/163] cached: Has any single infectious agent been identified as the cause of irritable bowel syndrome?
[3/163] cached: Is pollinosis another name for allergies to tree or grass pollen?
[4/163] cached: Which gene shapes the 3D enhancer network of the developing T cell genome?
[5/163] cached: Do mesenchymal-like glioblastoma cells show increased immune evasion?
[6/163] cached: Has Antibiotic use induced hiccups?
[7/163] cached: Is the use of surfactant restricted to neonatology?
[8/163] cached: What's the effect of liver RBFOX2?
[9/163] cached: Is there a shift in the cell states of glioblastoma during recurrence?
[10/163] cached: What is the difference in action between fomoterol and salmeterol?
[11/163] cached: Vector of dengue infection.
[12/163] cached: Which ensemble machine-learning framework has been developed harnessing UK biobank data?
[13/163] cached: What is a frequently lost chromosomal r

/tmp/ipykernel_57/3730426283.py:34: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  failed = answers[answers["query_error"].fillna("").ne("")]


## 2. Judge MyGPT answers against BioASQ ground truth

The judge handles yes/no ground truths and factoid answers (including BioASQ’s nested answer lists). It judges semantic equivalence, not exact string matching. Use the same `JUDGE_LLM` as the validated 03b pilot, if you set one there.

In [5]:
JUDGE_CLIENT = MyGPTClient.from_env(authenticate=False)
JUDGE_LLM = JUDGE_LLM or JUDGE_CLIENT.llm

JUDGE_PROMPT = """You are evaluating a biomedical question-answering system against the supplied BioASQ ground truth.
Judge only whether the system answer correctly answers this question according to the reference. Do not use outside knowledge to override the reference.

The reference may be:
- yes/no: the answer must express the same yes/no conclusion;
- a factoid or nested list of factoid answers: accept an equivalent name, abbreviation, or clear paraphrase. If the question asks for multiple items, all requested items must be present. Do not require exact wording.

Label 1 if the answer gives the correct conclusion or requested fact(s), consistent with the reference. Label 0 if it contradicts the reference, gives a wrong fact, omits a required part, or does not answer. If the answer is ambiguous, unsupported, or evasive, label 0.

QUESTION: {question}
BIOASQ GROUND TRUTH: {reference}
MYGPT ANSWER: {answer}

Return JSON only: {{"label": 0 or 1, "reason": "brief explanation"}}"""

def judge_answer(row):
    raw = JUDGE_CLIENT.ollama(
        JUDGE_PROMPT.format(
            question=str(row.question),
            reference=str(row.reference_answer),
            answer=str(row.answer)[:3000],
        ),
        model=JUDGE_LLM,
        temperature=0.0,
        json_mode=True,
    )
    try:
        result = json.loads(raw)
        label = result.get("label")
        if type(label) is not int or label not in (0, 1):
            raise ValueError("Judge label must be integer 0 or 1")
        return label, str(result.get("reason", "")), raw
    except (json.JSONDecodeError, AttributeError, TypeError):
        match = re.search(r'"?label"?\s*:\s*([01])', raw)
        if not match:
            raise ValueError(f"Could not parse a 0/1 judge label: {raw}")
        return int(match.group(1)), "", raw


### Apply the judge to every successfully answered question

Judge results are checkpointed separately so an interrupted run can resume. If any MyGPT or judge calls fail, fix the cause and rerun the relevant cell; the final evaluation CSV is only written when every question has a valid answer and judge label.

In [6]:
if not ANSWERS_FILE.exists():
    raise FileNotFoundError(f"Run section 1 first; missing {ANSWERS_FILE}")
answers = pd.read_csv(ANSWERS_FILE)
if len(answers) != len(questions) or answers["bioasq_id"].astype(str).tolist() != questions["bioasq_id"].astype(str).tolist():
    raise ValueError("Answer rows do not match the BioASQ source; rerun section 1")
failed_queries = answers[answers["query_error"].fillna("").ne("")]
if not failed_queries.empty:
    raise RuntimeError(f"MyGPT still has failed questions; rerun section 1 first: {failed_queries['bioasq_id'].tolist()}")

previous_judgments = {}
if JUDGE_PROGRESS_FILE.exists():
    prior = pd.read_csv(JUDGE_PROGRESS_FILE)
    if "bioasq_id" in prior.columns and "judge_label" in prior.columns:
        previous_judgments = {str(row.bioasq_id): row.to_dict() for _, row in prior.iterrows()}

judged = []
for _, row in answers.iterrows():
    bioasq_id = str(row.bioasq_id)
    cached = previous_judgments.get(bioasq_id, {})
    cached_label = cached.get("judge_label")
    judge_error = cached.get("judge_error")
    cache_matches = str(cached.get("answer", "")) == str(row.answer) and str(cached.get("reference_answer", "")) == str(row.reference_answer)
    if pd.notna(cached_label) and int(cached_label) in (0, 1) and cache_matches and not (pd.notna(judge_error) and str(judge_error).strip()):
        judged.append({**row.to_dict(), "judge_label": int(cached_label),
                       "judge_reason": cached.get("judge_reason", ""), "judge_raw": cached.get("judge_raw", ""),
                       "judge_error": ""})
        continue

    try:
        label, reason, raw = judge_answer(row)
        record = {**row.to_dict(), "judge_label": label, "judge_reason": reason,
                  "judge_raw": raw, "judge_error": ""}
    except Exception as exc:
        record = {**row.to_dict(), "judge_label": pd.NA, "judge_reason": "",
                  "judge_raw": "", "judge_error": str(exc)}
        print(f"Judge failed for {bioasq_id}: {exc}")

    judged.append(record)
    pd.DataFrame(judged).to_csv(JUDGE_PROGRESS_FILE, index=False)
    if len(judged) % 10 == 0 or len(judged) == len(answers):
        print(f"judged {len(judged)}/{len(answers)}")

judged = pd.DataFrame(judged)
judged.to_csv(JUDGE_PROGRESS_FILE, index=False)
judge_failures = judged[judged["judge_error"].fillna("").ne("")]
if not judge_failures.empty:
    raise RuntimeError(f"Judge failed for {len(judge_failures)} questions; rerun this cell to retry: {judge_failures['bioasq_id'].tolist()}")


/tmp/ipykernel_57/3271258833.py:6: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  failed_queries = answers[answers["query_error"].fillna("").ne("")]


judged 10/163
judged 20/163
judged 30/163
judged 40/163
judged 50/163
judged 60/163
judged 70/163
judged 80/163
judged 90/163
judged 100/163
judged 110/163
judged 120/163
judged 130/163
judged 140/163
judged 150/163
judged 160/163
judged 163/163


## 3. Save the labeled dataset for Module 4

In [7]:
calibration = pd.read_csv(JUDGE_PROGRESS_FILE)
if len(calibration) != len(questions) or calibration["judge_label"].isna().any():
    raise ValueError("Calibration labels are incomplete; finish section 2 before saving")

calibration["label"] = calibration["judge_label"].astype(int)
calibration["label_source"] = "judge"
calibration = calibration.drop(columns=["judge_raw", "query_error", "judge_error"], errors="ignore")
calibration.to_csv(EVAL_FILE, index=False)
print(f"Saved {len(calibration)} labeled BioASQ rows to {EVAL_FILE}")
print("Labels (1=correct, 0=incorrect):")
print(calibration["label"].value_counts().sort_index().rename(index={0: "incorrect", 1: "correct"}))
calibration[["bioasq_id", "question", "reference_answer", "answer", "label", "judge_reason"]].head(10)


Saved 163 labeled BioASQ rows to /workspace/workshop_outputs/03_calibration_eval_dataset.csv
Labels (1=correct, 0=incorrect):
label
incorrect     53
correct      110
Name: count, dtype: int64


,bioasq_id,question,reference_answer,answer,label,judge_reason
0,67fc20ee18b1e36f2e00012a,Do glioblastoma tumors have a high degree of l...,no,"According to the provided context, glioblastom...",0,"The answer contradicts the reference, stating ..."
1,67ed650c18b1e36f2e0000fe,Has any single infectious agent been identifie...,no,"Based on the provided context, the answer is:\...",0,The answer does not provide a clear 'yes' or '...
2,67daf71018b1e36f2e000057,Is pollinosis another name for allergies to tr...,yes,The library contains the answer!\n\nAccording ...,0,The answer is incorrect and does not provide a...
3,67d4903418b1e36f2e000018,Which gene shapes the 3D enhancer network of t...,[['SATB1']],"According to the provided context, the gene th...",1,The answer correctly identifies SATB1 as the g...
4,67f6f14118b1e36f2e000100,Do mesenchymal-like glioblastoma cells show in...,yes,"Based on the provided context, it can be concl...",1,The answer correctly concludes that mesenchyma...
5,67fc546c18b1e36f2e000130,Has Antibiotic use induced hiccups?,yes,The answer is: Yes.\n\nThe context contains se...,1,The answer correctly concludes that antibiotic...
6,67d6c10618b1e36f2e000027,Is the use of surfactant restricted to neonato...,no,"Based on the provided context, the answer is:\...",1,The answer correctly concludes that the use of...
7,67d497c418b1e36f2e00001a,What's the effect of liver RBFOX2?,[['Liver RBFOX2 regulates cholesterol homeosta...,"According to the provided context, the effect ...",1,"The answer correctly answers the question, pro..."
8,67da131f18b1e36f2e000053,Is there a shift in the cell states of gliobla...,yes,"According to the provided context, the answer ...",1,The answer correctly concludes that there is a...
9,67f8ea8318b1e36f2e000107,What is the difference in action between fomot...,[['Formoterol has a faster onset of action com...,"According to the provided context, the main di...",1,The answer correctly identifies the difference...


## Notes

- This source file is called `questions_all_164.csv` but currently has 163 complete rows; the notebook uses all rows present and does not invent an extra question.
- The dataset contains on-topic BioASQ questions, not injected irrelevant questions. Module 4 uses incorrect judge labels as the positive class for weight calibration; check the resulting class counts before interpreting PR-AUC.
- The judge progress file is `workshop_outputs/03_calibration_judge_progress.csv`; the finished dataset for Module 4 is `workshop_outputs/03_calibration_eval_dataset.csv`.

➡️ Next: use the calibration dataset in Module 4.